# Turkish Morphology Retrieval — Paper Evaluation

Bu notebook v3 benchmark için iki ayrı görevi raporlar:

1. **Closed contrast:** her query yalnız kendi 11 adayıyla (1 positive, 8 hard, 2 easy).
2. **Pooled full corpus:** 500 final query, en fazla 5.500 ortak doküman ve insan-pooling qrels.

Artefakt baseline'ları gold rolünü girdi olarak kullanmaz. Full-corpus hücresi yalnız own-gold qrels ile çalışmayı reddeder; yabancı dokümanlar pooled human judgment almadan negatif kabul edilmez.

## 1. Setup

In [ ]:
%pip -q install sentence-transformers scikit-learn pandas matplotlib

import gc, json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if not (ROOT / 'test' / 'evaluation.py').exists():
    for parent in [ROOT, *ROOT.parents]:
        if (parent / 'test' / 'evaluation.py').exists():
            ROOT = parent
            break
sys.path.insert(0, str(ROOT))

from sentence_transformers import SentenceTransformer
from test.evaluation import (
    ablate_items, approximate_randomization, bootstrap_ci, candidate_only_classifier,
    closed_qrels, evaluate_artifacts, evaluate_run, holm_adjust, load_items, load_qrels,
    mcnemar, paired_bootstrap, score_encoder, slice_summary,
)
print('root:', ROOT)
print('cuda:', torch.cuda.is_available())

## 2. Frozen data
Sealed internal dosyayı yalnız nihai model dondurulduktan sonra açın.

In [ ]:
RUN_ID = 'test_v31'
DEV_FILE = ROOT / 'test' / 'runs' / RUN_ID / 'release' / 'morph_dev_v3.1.0.json'
TEST_INTERNAL_FILE = ROOT / 'test' / 'runs' / RUN_ID / 'private' / 'morph_test_internal_v3.1.0.json'

if not DEV_FILE.exists() or not TEST_INTERNAL_FILE.exists():
    raise FileNotFoundError('RUN_ID veya frozen dosya yollarını ayarlayın; blind test dosyası metrik için yeterli değildir.')
DEV = load_items(DEV_FILE)
TEST = load_items(TEST_INTERNAL_FILE)
assert len(DEV) == 100 and len(TEST) == 500
assert all(len(item['candidates']) == 11 for item in DEV + TEST)
print('development:', len(DEV), 'sealed test:', len(TEST), 'test docs:', 11 * len(TEST))

## 3. Query-blind / cheap artifact baselines

In [ ]:
artifact_rows = evaluate_artifacts(DEV, TEST)
artifact_rows['candidate_only_char_tfidf'] = candidate_only_classifier(DEV, TEST)['summary']
display(pd.DataFrame(artifact_rows).T.sort_values('recall@1', ascending=False).round(4))
print('Closed-set random R@1:', round(1/11, 4))
print('Not: role etiketi baseline girdisi değildir; position-only konumu yalnız development setinden öğrenir.')

## 4. Encoder closed-contrast evaluation

In [ ]:
INSTRUCT = 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: '
MODELS = [
    ('e5-large', 'intfloat/multilingual-e5-large', 'query: ', 'passage: ', {}),
    ('bge-m3', 'BAAI/bge-m3', '', '', {}),
    ('modernbert-tr', 'ytu-ce-cosmos/modernbert-tr-embed', INSTRUCT, '', {}),
    ('qwen3-8b', 'Qwen/Qwen3-Embedding-8B', INSTRUCT, '', {'model_kwargs': {'torch_dtype': torch.float16}}),
]
RESULTS = {}
for name, repo, query_prefix, document_prefix, kwargs in MODELS:
    print(name, 'loading...')
    model = SentenceTransformer(repo, trust_remote_code=True, **kwargs)
    try:
        model.default_prompt_name = None
    except Exception:
        pass
    RESULTS[name] = score_encoder(model, TEST, query_prefix, document_prefix, full_corpus=False, batch_size=16)
    del model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
summary = pd.DataFrame({name: result['summary'] for name, result in RESULTS.items()}).T
display(summary.sort_values('recall@1', ascending=False).round(4))

## 5. Query-level bootstrap CI and diagnostic slices

In [ ]:
ci_rows = []
for model_name, result in RESULTS.items():
    for metric in ['hard_only_recall@1', 'contrast_consistency', 'recall@1', 'recall@5', 'mrr@10', 'ndcg@10']:
        values = [row[metric] for row in result['per_query']]
        low, high = bootstrap_ci(values, seed=20260814)
        ci_rows.append({'model': model_name, 'metric': metric, 'mean': np.mean(values), 'ci_low': low, 'ci_high': high})
display(pd.DataFrame(ci_rows).round(4))

FOCUS = summary['recall@1'].idxmax()
slices = slice_summary(RESULTS[FOCUS]['per_query'], TEST, metric='recall@1')
for field, values in slices.items():
    print('---', field, '---')
    display(pd.DataFrame(values).T.sort_values('mean').round(4))

## 6. Paired model comparisons
McNemar yalnız binary R@1 içindir; rank metriklerinde paired bootstrap/randomization kullanılır.

In [ ]:
REFERENCE = 'e5-large'
comparisons, raw_p = [], {}
ref = {row['query_id']: row for row in RESULTS[REFERENCE]['per_query']}
for model_name, result in RESULTS.items():
    if model_name == REFERENCE: continue
    cur = {row['query_id']: row for row in result['per_query']}
    qids = sorted(ref)
    left = [cur[q]['recall@1'] for q in qids]
    right = [ref[q]['recall@1'] for q in qids]
    boot = paired_bootstrap(left, right, seed=20260814)
    p = approximate_randomization(left, right, seed=20260814)
    raw_p[model_name] = p
    comparisons.append({'model': model_name, 'vs': REFERENCE, **boot, **mcnemar(left, right), 'randomization_p': p})
adjusted = holm_adjust(raw_p)
for row in comparisons: row['holm_p'] = adjusted[row['model']]
display(pd.DataFrame(comparisons).round(5))

## 7. Morphology ablations
Kritik sözcük silme ve F5-kök proxy değerlendirmesi, model başarısının suffix sinyalinden ne kadar yararlandığını ölçer.

In [ ]:
focus_spec = next(spec for spec in MODELS if spec[0] == FOCUS)
name, repo, query_prefix, document_prefix, kwargs = focus_spec
model = SentenceTransformer(repo, trust_remote_code=True, **kwargs)
ABLATIONS = {'original': TEST, 'critical_deleted': ablate_items(TEST, 'critical_deleted'), 'f5_roots': ablate_items(TEST, 'f5_roots')}
ablation_results = {}
for ablation_name, items in ABLATIONS.items():
    ablation_results[ablation_name] = score_encoder(model, items, query_prefix, document_prefix)['summary']
del model
display(pd.DataFrame(ablation_results).T.round(4))

## 8. Pooled full-corpus evaluation
Bu hücre için qrels, pool içindeki yargılanmış negatifleri `0`, kısmi/tam alakalıları `1/2` olarak içermelidir. Own-gold-only dosya kabul edilmez.

In [ ]:
POOLED_QRELS = ROOT / 'test' / 'runs' / RUN_ID / 'private' / 'pooled_human_qrels.tsv'
if not POOLED_QRELS.exists():
    print('SKIP: BM25/char-ngram/dense/reranker pool insan yargıları henüz yok.')
else:
    pooled_qrels = load_qrels(POOLED_QRELS)
    judged_rows = sum(len(rels) for rels in pooled_qrels.values())
    if judged_rows <= len(TEST) or not any(score == 0 for rels in pooled_qrels.values() for score in rels.values()):
        raise ValueError('Bu own-gold-only qrels görünüyor; full-corpus paper metriği hesaplanamaz.')
    FULL = {}
    for name, repo, query_prefix, document_prefix, kwargs in MODELS:
        model = SentenceTransformer(repo, trust_remote_code=True, **kwargs)
        FULL[name] = score_encoder(model, TEST, query_prefix, document_prefix, full_corpus=True, qrels=pooled_qrels, batch_size=16)
        del model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    display(pd.DataFrame({name: result['summary'] for name, result in FULL.items()}).T.round(4))

## 9. Fine-tuning seed reporting
Her fine-tuned konfigürasyon en az 3, tercihen 5 seed ile çalıştırılır. Ana tabloda seed ortalaması ± standart sapma; model farkında query-paired test raporlanır.

In [ ]:
# Her seed'in per-query çıktısını `model`, `seed`, `query_id`, `recall@1`, `mrr@10`, `ndcg@10`
# kolonlarıyla birleştirdikten sonra kullanın. Teste göre seed seçmeyin.
SEED_RESULTS_CSV = None
if SEED_RESULTS_CSV:
    seed_df = pd.read_csv(SEED_RESULTS_CSV)
    display(seed_df.groupby(['model', 'seed'])[['recall@1', 'mrr@10', 'ndcg@10']].mean()
            .groupby('model').agg(['mean', 'std']).round(4))

## 10. Export

In [ ]:
summary.to_csv('closed_encoder_summary.csv')
pd.DataFrame(ci_rows).to_csv('closed_bootstrap_ci.csv', index=False)
pd.DataFrame(comparisons).to_csv('paired_model_tests.csv', index=False)
pd.DataFrame(artifact_rows).T.to_csv('artifact_baselines.csv')
print('Exported paper tables. Full-corpus tables are exported only after pooled human qrels exist.')